In [1]:
from pathlib import Path

DATA_DIR = Path("../data/metrics/method-comparison")

algorithms = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])

algorithms

['calgan-aesop',
 'calgan-l1',
 'unet-aesop',
 'unet-l1',
 'vgg128-aesop',
 'vgg128-l1']

In [2]:
datasets = sorted(set(
    p.stem.replace("_metrics", "")
    for alg in algorithms for p in (DATA_DIR / alg).glob("*_metrics.csv")
))
datasets

['bsd100', 'manga109', 'set14', 'set5', 'urban100']

In [24]:

import pandas as pd

METRICS = ['psnr', 'ssim', 'lpips', 'dists', 'niqe']
HIGHER_BETTER = {'psnr', 'ssim'}

frames = []
for alg in algorithms:
    for ds in datasets:
        csv_path = DATA_DIR / alg / f"{ds}_metrics.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df['algorithm'] = alg
            df['dataset'] = ds
            df = df[['algorithm', 'dataset', 'image_name', *METRICS]]
            frames.append(df)

all_data = pd.concat(frames, ignore_index=True)

all_data

,algorithm,dataset,image_name,psnr,ssim,lpips,dists,niqe
0,calgan-aesop,bsd100,bsd100_0_0,24.0386,0.5624,0.1972,0.1338,4.2122
1,calgan-aesop,bsd100,bsd100_1_1,23.7181,0.6425,0.1901,0.1237,4.4914
2,calgan-aesop,bsd100,bsd100_2_2,26.5694,0.7386,0.1700,0.1020,4.2828
3,calgan-aesop,bsd100,bsd100_3_3,29.8018,0.7632,0.1809,0.1446,5.5914
4,calgan-aesop,bsd100,bsd100_4_4,24.4116,0.6420,0.2027,0.1654,5.6169
...,...,...,...,...,...,...,...,...
1963,vgg128-l1,urban100,urban100_95_95,31.4115,0.9065,0.0367,0.0523,5.2746
1964,vgg128-l1,urban100,urban100_96_96,22.3755,0.5944,0.1770,0.0996,1.9438
1965,vgg128-l1,urban100,urban100_97_97,20.4373,0.5776,0.1567,0.0728,2.8076
1966,vgg128-l1,urban100,urban100_98_98,29.8774,0.8836,0.0555,0.0536,3.7977


In [25]:
from scipy import stats

ALPHA = 0.05

results = []
for alg in algorithms:
    for ds in datasets:
        ds_data = all_data[(all_data['algorithm'] == alg) & (all_data['dataset'] == ds)]
        if len(ds_data) < 3:  # Shapiro-Wilk requires at least 3 samples
            continue
        for m in METRICS:
            data = ds_data[m].values
            # If all values are identical, Shapiro-Wilk fails
            if len(set(data)) <= 1:
                p = 1.0
            else:
                _, p = stats.shapiro(data)

            is_normal = p > ALPHA
            results.append({
                'algorithm': alg,
                'dataset': ds,
                'metric': m,
                'p-value': p,
                'is_normal': is_normal
            })

df_normality = pd.DataFrame(results)

# SUMMARY
total_tests = len(df_normality)
normal_count = df_normality['is_normal'].sum()
non_normal_count = total_tests - normal_count

print(f"Normality Test Summary (Shapiro-Wilk, alpha={ALPHA}):")
print(f"Total distributions tested: {total_tests}")
print(f"Normal (p > 0.05):       {normal_count} ({normal_count / total_tests:.1%})")
print(f"Non-Normal (p <= 0.05):   {non_normal_count} ({non_normal_count / total_tests:.1%})")

df_normality

Normality Test Summary (Shapiro-Wilk, alpha=0.05):
Total distributions tested: 150
Normal (p > 0.05):       64 (42.7%)
Non-Normal (p <= 0.05):   86 (57.3%)


,algorithm,dataset,metric,p-value,is_normal
0,calgan-aesop,bsd100,psnr,2.869292e-01,True
1,calgan-aesop,bsd100,ssim,1.198903e-02,False
2,calgan-aesop,bsd100,lpips,1.587655e-05,False
3,calgan-aesop,bsd100,dists,4.148540e-02,False
4,calgan-aesop,bsd100,niqe,6.169549e-08,False
...,...,...,...,...,...
145,vgg128-l1,urban100,psnr,2.518903e-02,False
146,vgg128-l1,urban100,ssim,3.737542e-04,False
147,vgg128-l1,urban100,lpips,1.682076e-04,False
148,vgg128-l1,urban100,dists,1.531278e-05,False


In [26]:
from itertools import combinations

method_pairs = list(combinations(algorithms, 2))

method_pairs

[('calgan-aesop', 'calgan-l1'),
 ('calgan-aesop', 'unet-aesop'),
 ('calgan-aesop', 'unet-l1'),
 ('calgan-aesop', 'vgg128-aesop'),
 ('calgan-aesop', 'vgg128-l1'),
 ('calgan-l1', 'unet-aesop'),
 ('calgan-l1', 'unet-l1'),
 ('calgan-l1', 'vgg128-aesop'),
 ('calgan-l1', 'vgg128-l1'),
 ('unet-aesop', 'unet-l1'),
 ('unet-aesop', 'vgg128-aesop'),
 ('unet-aesop', 'vgg128-l1'),
 ('unet-l1', 'vgg128-aesop'),
 ('unet-l1', 'vgg128-l1'),
 ('vgg128-aesop', 'vgg128-l1')]

In [32]:
rows = []

for a, b in pairs:
    for dataset in datasets:
        a_data = all_data[(all_data['algorithm'] == a) & (all_data['dataset'] == dataset)].sort_values('image_name')
        b_data = all_data[(all_data['algorithm'] == b) & (all_data['dataset'] == dataset)].sort_values('image_name')

        for m in METRICS:
            a_score = a_data[m].values
            b_score = b_data[m].values
            _, p = stats.wilcoxon(a_score, b_score)
            #print(f"{a} vs {b}: {m} p-value: {p}")
            diff = a_score.mean() - b_score.mean()
            winner = None

            if m in HIGHER_BETTER:
                winner = a if diff > 0 else b
            else:
                winner = a if diff < 0 else b

            rows.append(
                {'dataset': dataset, 'metric': m, 'algorithm_a': a, 'algorithm_b': b, 'p-value': p, 'better': winner}
            )

posthoc_df = pd.DataFrame(rows)
posthoc_df

,dataset,metric,algorithm_a,algorithm_b,p-value,better
0,bsd100,psnr,calgan-aesop,calgan-l1,3.571611e-16,calgan-aesop
1,bsd100,ssim,calgan-aesop,calgan-l1,4.810192e-18,calgan-aesop
2,bsd100,lpips,calgan-aesop,calgan-l1,3.895681e-18,calgan-l1
3,bsd100,dists,calgan-aesop,calgan-l1,2.632423e-17,calgan-l1
4,bsd100,niqe,calgan-aesop,calgan-l1,3.010681e-16,calgan-l1
...,...,...,...,...,...,...
370,urban100,psnr,vgg128-aesop,vgg128-l1,2.543747e-09,vgg128-aesop
371,urban100,ssim,vgg128-aesop,vgg128-l1,7.133920e-12,vgg128-aesop
372,urban100,lpips,vgg128-aesop,vgg128-l1,3.632468e-04,vgg128-aesop
373,urban100,dists,vgg128-aesop,vgg128-l1,1.935563e-02,vgg128-aesop


In [37]:
filtered_posthoc_df = posthoc_df[posthoc_df['p-value'] <= ALPHA]

wins = filtered_posthoc_df['better'].value_counts().reindex(algorithms, fill_value=0)
wins

losses = pd.Series(0, index=algorithms)
losses

for _, r in filtered_posthoc_df.iterrows():
    loser = r['algorithm_b'] if r['better'] == r['algorithm_a'] else r['algorithm_a']
    losses[loser] += 1

leaderboard = pd.DataFrame({'Wins': wins, 'Losses': losses, 'Net': wins - losses})
leaderboard = leaderboard.sort_values('Net', ascending=False)
leaderboard

,Wins,Losses,Net
unet-aesop,54,30,24
vgg128-aesop,52,29,23
vgg128-l1,45,38,7
unet-l1,48,42,6
calgan-l1,40,47,-7
calgan-aesop,17,70,-53
